# Evaluate nanoVLM checkpoints on MiniGrid

Loads checkpoints saved by `sft_train.ipynb` and evaluates success rate.
The model stays in fp32 (same as during training).

In [ ]:
import sys
sys.path.append("nanoVLM")

import json
import os
import glob

import torch
from PIL import Image

from models.vision_language_model import VisionLanguageModel
from data.processors import get_image_processor
from env_utils import (
    create_env, get_agent_view, randomize_positions,
    ACTION_NAMES, text_to_action,
)
from expert import bfs_path

print("Imports OK")

In [ ]:
# ---------- Config ----------
CHECKPOINTS_DIR = "checkpoints"
EPISODES = 50            # episodes per checkpoint
MAX_STEPS = 100          # max steps per episode

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# ---------- Load base model ----------
print("Loading nanoVLM-460M-8k...")
model = VisionLanguageModel.from_pretrained("lusxvr/nanoVLM-460M-8k")
model.to(device=device)
model.eval()
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

tokenizer = model.tokenizer
image_processor = get_image_processor(
    max_img_size=model.cfg.max_img_size,
    splitted_image_size=model.cfg.vit_img_size,
    resize_to_max_side_len=model.cfg.resize_to_max_side_len,
)

In [ ]:
# ---------- Helpers ----------
def make_image_string(tokenizer, n_h, n_w, mp_len):
    parts = []
    if hasattr(tokenizer, "global_image_token"):
        parts.append(tokenizer.global_image_token)
        parts.append(tokenizer.image_token * mp_len)
        if n_h == 1 and n_w == 1:
            return "".join(parts)
    for i in range(n_h):
        for j in range(n_w):
            parts.append(getattr(tokenizer, f"r{i+1}c{j+1}"))
            parts.append(tokenizer.image_token * mp_len)
    return "".join(parts)


def parse_action(text):
    text = text.strip().lower()
    for name in ACTION_NAMES:
        if name in text:
            return text_to_action(name)
    return None

In [ ]:
@torch.inference_mode()
def evaluate(model, tokenizer, image_processor, num_episodes=100, max_steps=100):
    env = create_env()
    mp_len = model.cfg.mp_image_token_length
    prompt = (
        "What is the next action to reach the green goal? "
        "Choose from: turn left, turn right, move forward."
    )

    successes = 0
    total_steps = 0
    oracle_steps = 0

    for ep in range(num_episodes):
        env.reset()
        randomize_positions(env)
        oracle = bfs_path(env)
        oracle_len = len(oracle) if oracle else 0
        oracle_steps += oracle_len

        done = False
        truncated = False
        step = 0

        while not (done or truncated) and step < max_steps:
            img = get_agent_view(env)
            processed_img, (n_h, n_w) = image_processor(img)

            image_string = make_image_string(tokenizer, n_h, n_w, mp_len)
            messages = [
                {"role": "user", "content": image_string + prompt},
            ]

            input_ids = tokenizer.apply_chat_template(
                messages, tokenize=True, add_special_tokens=False
            )
            input_ids = torch.tensor([input_ids], dtype=torch.long, device=device)
            attention_mask = torch.ones_like(input_ids)

            generated_ids = model.generate(
                input_ids,
                images=[processed_img.to(device=device)],
                attention_mask=attention_mask,
                max_new_tokens=10,
                top_k=1,
                greedy=True,
            )

            generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
            action = parse_action(generated_text)

            if action is None:
                action = 2

            _, reward, done, truncated, _ = env.step(action)
            step += 1

        if done:
            successes += 1

        total_steps += step
        print(
            f"Ep {ep + 1}/{num_episodes}: "
            f"{'SUCCESS' if done else 'FAIL'} "
            f"({step}/{oracle_len} steps)"
        )

    success_rate = successes / num_episodes * 100
    avg_steps = total_steps / num_episodes
    avg_oracle = oracle_steps / num_episodes
    print(f"\nSuccess rate: {success_rate:.1f}%")
    print(f"Avg steps: {avg_steps:.1f} (oracle: {avg_oracle:.1f})")

    return success_rate, avg_steps, avg_oracle

In [ ]:
# ---------- Evaluate all epoch checkpoints ----------
ckpt_pattern = os.path.join(CHECKPOINTS_DIR, "sft_epoch_*.pt")
ckpt_paths = sorted(
    glob.glob(ckpt_pattern),
    key=lambda p: int(p.split("_epoch_")[1].split(".pt")[0]),
)
print(f"Found {len(ckpt_paths)} epoch checkpoints in {CHECKPOINTS_DIR}/")

results = []
for ckpt_path in ckpt_paths:
    epoch = int(ckpt_path.split("_epoch_")[1].split(".pt")[0])
    print(f"\n{'='*60}")
    print(f"Evaluating epoch {epoch}...")
    print(f"{'='*60}")
    model.load_state_dict(torch.load(ckpt_path, map_location=device), strict=False)
    model.eval()
    sr, avg_s, avg_o = evaluate(model, tokenizer, image_processor, EPISODES, MAX_STEPS)
    results.append({"epoch": epoch, "success_rate": sr, "avg_steps": avg_s, "avg_oracle": avg_o})

In [ ]:
# ---------- Evaluate final model ----------
final_path = os.path.join(CHECKPOINTS_DIR, "sft_model.pt")
if os.path.exists(final_path):
    print(f"\n{'='*60}")
    print("Evaluating final model...")
    print(f"{'='*60}")
    model.load_state_dict(torch.load(final_path, map_location=device), strict=False)
    model.eval()
    sr, avg_s, avg_o = evaluate(model, tokenizer, image_processor, EPISODES, MAX_STEPS)
    results.append({"epoch": "final", "success_rate": sr, "avg_steps": avg_s, "avg_oracle": avg_o})
else:
    print(f"No final model found at {final_path}, skipping.")

In [ ]:
# ---------- Print summary ----------
print("\n\n=== SUMMARY ===")
print(f"{'Epoch':<8} {'Success Rate':<16} {'Avg Steps':<12} {'Oracle Steps':<14}")
print("-" * 50)
for r in results:
    epoch_str = str(r['epoch'])
    print(f"{epoch_str:<8} {r['success_rate']:<16.1f} {r['avg_steps']:<12.1f} {r['avg_oracle']:<14.1f}")

In [ ]:
# ---------- Save results ----------
out_path = os.path.join(CHECKPOINTS_DIR, "eval_results.json")
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {out_path}")

In [ ]:
# ---------- Plot learning curves ----------
import matplotlib.pyplot as plt

epoch_results = [r for r in results if r["epoch"] != "final"]
epochs = [r["epoch"] for r in epoch_results]
final_result = [r for r in results if r["epoch"] == "final"]
avg_oracle = results[0]["avg_oracle"] if results else 0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Success rate
ax1.plot(epochs, [r["success_rate"] for r in epoch_results],
         marker="o", label="Model")
ax1.axhline(y=100, color="green", linestyle="--", alpha=0.7, label="Oracle (100%)")
if final_result:
    ax1.axhline(y=final_result[0]["success_rate"], color="blue", linestyle=":", alpha=0.5,
                label=f"Final: {final_result[0]['success_rate']:.1f}%")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Success Rate (%)")
ax1.set_title("Success Rate vs Epoch")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Avg steps
ax2.plot(epochs, [r["avg_steps"] for r in epoch_results],
         marker="o", label="Model")
ax2.axhline(y=avg_oracle, color="green", linestyle="--", alpha=0.7,
            label=f"Oracle ({avg_oracle:.1f})")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Avg Steps to Goal")
ax2.set_title("Avg Steps vs Epoch")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(CHECKPOINTS_DIR, "eval_curves.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {plot_path}")